# Assignment 8
### Data source: 20 news group from sklearn
### Objective: classify news article categories
### Classification methods: SGDClassifier vs. Roberta(transformer)

In [3]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from xgboost import XGBClassifier
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from datasets import Dataset
from torch.optim import AdamW

In [4]:
# Removing headers, footers, and quotes makes the task much more realistic
data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
X, y = data.data, data.target

We will preview the dataset structure and length:

In [5]:
print(f'Length of the articles: {len(X)}')
print(f'Length of y: {len(y)}')

Length of the articles: 18846
Length of y: 18846


Split the dataset into train and test 80/20 ratio:

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'X Train size: {len(X_train)}')
print(f'y Train size: {len(y_train)}')
print(f'X Test size: {len(X_test)}')
print(f'y Test size: {len(y_test)}')


X Train size: 15076
y Train size: 15076
X Test size: 3770
y Test size: 3770


Tokenization of the text into vectors:

In [7]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


Pass the vectorized features into SGDClassifier (Essentially same as PAC) for training

In [8]:
#Define the model:
SGD = SGDClassifier(loss='modified_huber', penalty='l2', alpha=1e-4, random_state=42)

#fit the pac model:
SGD_model = SGD.fit(X_train_tfidf, y_train)

SGD_pred = SGD_model.predict(X_test_tfidf)

#Print out the metrics:
accuracy = accuracy_score(y_test, SGD_pred)
print(f'SGD model accuracy: {accuracy}')
print('Classification report:')
print(classification_report(SGD_pred, y_test, target_names=data.target_names))

SGD model accuracy: 0.7551724137931034
Classification report:
                          precision    recall  f1-score   support

             alt.atheism       0.61      0.68      0.64       135
           comp.graphics       0.73      0.81      0.77       182
 comp.os.ms-windows.misc       0.69      0.70      0.69       194
comp.sys.ibm.pc.hardware       0.70      0.63      0.66       205
   comp.sys.mac.hardware       0.71      0.79      0.75       184
          comp.windows.x       0.82      0.83      0.83       211
            misc.forsale       0.73      0.75      0.74       186
               rec.autos       0.80      0.51      0.62       307
         rec.motorcycles       0.74      0.81      0.77       155
      rec.sport.baseball       0.84      0.93      0.88       192
        rec.sport.hockey       0.89      0.94      0.92       188
               sci.crypt       0.80      0.86      0.83       188
         sci.electronics       0.72      0.72      0.72       201
             

The results from SDG algorithm is very decent. The processing time is also instant with overall macro average of 0.75 for 20 different labels.

RoBerta implementation:

In [9]:
# Define the dataset parameters:
text = data.data
labels = data.target.tolist()
label_names = data.target_names
num_labels = len(set(labels))

Define the dataset format using Huggingface dataset class:

In [17]:
# Tokenize using .map()
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256)


#Use the same train and test sets from above:
train_dataset = Dataset.from_dict({"text": X_train, "labels": y_train.tolist()})
test_dataset = Dataset.from_dict({"text": X_test, "labels": y_test.tolist()})

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

#Feed directly into DataLoader:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16)

Map:   0%|          | 0/15076 [00:00<?, ? examples/s]

Map:   0%|          | 0/3770 [00:00<?, ? examples/s]

Loading the model:

In [18]:
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=num_labels)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #Use GPU if available
model.to(device)
print(f'Using device: {device}')


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


Optimizer & Scheduler

In [19]:
EPOCHS = 5
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(), lr = LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

Training loop

In [20]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_batch = batch['labels'].to(device)
        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  
        optimizer.step()                                                    
        scheduler.step()                                                    
        total_loss += loss.item()                                           
    return total_loss / len(loader)

for epoch in range(EPOCHS):
    avg_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    print(f"Epoch {epoch + 1} / {EPOCHS} | avg loss: {avg_loss:.4f}")

Epoch 1 / 5 | avg loss: 1.6347
Epoch 2 / 5 | avg loss: 0.8368
Epoch 3 / 5 | avg loss: 0.6048
Epoch 4 / 5 | avg loss: 0.4468
Epoch 5 / 5 | avg loss: 0.3442


Evaluation & Report

In [21]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_batch = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().tolist()) #Bring it back to cpu from gpu.
            all_labels.extend(labels_batch.cpu().tolist())
    return all_preds, all_labels

preds, true = evaluate(model, test_loader, device)
roberta_accuracy = accuracy_score(true, preds)
print(f"RoBERTa model accuracy: {roberta_accuracy}")
print("Classification report:")
print(classification_report(true, preds, target_names=label_names))


RoBERTa model accuracy: 0.7472148541114059
Classification report:
                          precision    recall  f1-score   support

             alt.atheism       0.52      0.53      0.53       151
           comp.graphics       0.74      0.72      0.73       202
 comp.os.ms-windows.misc       0.61      0.72      0.66       195
comp.sys.ibm.pc.hardware       0.62      0.70      0.65       183
   comp.sys.mac.hardware       0.85      0.68      0.76       205
          comp.windows.x       0.85      0.88      0.86       215
            misc.forsale       0.83      0.78      0.80       193
               rec.autos       0.60      0.76      0.67       196
         rec.motorcycles       0.81      0.74      0.78       168
      rec.sport.baseball       0.89      0.87      0.88       211
        rec.sport.hockey       0.90      0.89      0.90       198
               sci.crypt       0.79      0.77      0.78       201
         sci.electronics       0.69      0.69      0.69       202
         

Conclusion: In the 20 news dataset, the SDG classification performed exceptionally well when compared to a transformer-based, Roberta, model. The overall macro average is higher than the ones observed in the Roberta model (0.75 vs. 0.73, for SDG and Roberta, respectively). More importantly, the processing time in SDG was also instantaneous while the Roberta model with 5 epochs, which was still not at convergence, took 30+mins to loop. There are several potential reasons why the SDG is better suited in this case. 1) news classification is basically a bag-of-word situation, meaning that the accuracy of the classification of the news articles depends on specific words present in the text. While Roberta is better suited for syntax, context, or semantic classifications, SDG has the advantage to identify the key words in the text and product the classification. 2) the current setting of 5 epochs is not enough to reach convergence in the model. This is shown by the average loss in the fifth epoch, which was only reducing the loss by roughly 10% from the previous run, indicting more runs can be applied to further reduce the average loss. However, this was challenging since utilizing PyTorch was computationally heavy even with an Nvidia GPU. Nevertheless, in this project, we demonstrated that there is no one-tool-does-it-all method. Each dataset requires understanding of the context, structure, and the computational environment for the optimal model selection.